# 真实本地模型：DoLa 层间对比解码

在真实本地因果模型上运行 DoLa：每个解码步按 Jensen-Shannon 散度动态选择早期层，放大深层事实信号。报告同时给出常规解码基线与缓解后回答。

## 真实集成前提

安装：`python3 -m pip install -e ".[hf]"`；需要已下载的本地模型目录。CLI 等价命令：`hallucination-mitigation run --method dola --model <LOCAL_MODEL> --prompt ...`。

代码单元没有预执行。填写本地模型路径或 API 凭据后，从上到下运行；不要将 API key 保存到 notebook。缓解会改变模型输出，务必同时阅读基线与缓解后的回答。

## 参数说明

- `LOCAL_MODEL: str`：本地 causal LM 目录（Llama/Qwen 系）。
- `relative_top: float = 0.1`：自适应合理性约束阈值，过滤低概率候选。
- `mature_layer / candidate_premature_layers`：成熟层与候选早期层桶；缺省按层数自动选取。
- `MAX_NEW_TOKENS: int = 64`：生成长度。

## 步骤 1：配置真实模型

In [ ]:
from pathlib import Path
import sys
repo_root = Path.cwd()
if not (repo_root / "src" / "hallucination").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

LOCAL_MODEL = "/absolute/path/to/your/causal-lm"   # e.g. a local Qwen2.5-1.5B-Instruct directory
DEVICE = "cpu"     # "cuda" when a GPU is available
DTYPE = "auto"
MAX_NEW_TOKENS = 64
if not Path(LOCAL_MODEL).is_dir():
    raise ValueError("Set LOCAL_MODEL to an existing local model directory (download it first).")

## 步骤 2：构建 Mitigator

In [ ]:
from hallucination.mitigation import Mitigator
mitigator = Mitigator(
    method="dola",
    model=LOCAL_MODEL,
    device=DEVICE,
    dtype=DTYPE,
    local_files_only=True,
    relative_top=0.1,
    baseline=True,
)

## 步骤 3：交互运行并读取

In [ ]:
prompt = input("Prompt：").strip()
report = mitigator.mitigate(prompt, max_new_tokens=MAX_NEW_TOKENS)
print("baseline :", report.baseline)
print("mitigated:", report.response)
print("premature layer distribution:", report.trace["premature_layer_distribution"])
print("warnings:", report.warnings)

## 结果解读与限制

DoLa 只改变解码分布，不保证事实正确；层桶是任务相关的预置参数，必须在自己的模型与数据上验证，真实冒烟中出现过把正确基线改错的情况。